In [1]:
r"""
=================================================================================
ORALVISION-M4 — RTX 2050 HARDENED VERSION
EfficientNet-B0 + Coordinate Attention + Logistic Regression head
CLEAN PRE-SPLIT DATASET + EXTENDED EVALUATION
=================================================================================

Dataset root:
    C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN

IMPORTANT:
    This version expects the CLEAN dataset to already contain separate
    train / val (or validation) / test directories. It does NOT perform
    a random split and it does NOT augment validation/test data.

RTX 2050 hardening:
  - Automatic CUDA detection
  - AMP / mixed precision
  - REAL batch size 16
  - Gradient accumulation x2 (effective batch size 32)
  - num_workers=0 for Windows/Jupyter stability
  - CUDA allocator fragmentation mitigation
  - GPU memory cleanup
  - Per-epoch atomic checkpoints
  - Safe RNG save/restore
  - Resume after kernel crash/interruption
  - Best-model preservation
  - Early stopping
  - ReduceLROnPlateau
  - CUDA OOM checkpoint + clear + actionable message
  - Explicit GPU memory reporting

Model:
  EfficientNet-B0 (ImageNet pretrained)
      -> Coordinate Attention
      -> Global Average Pooling
      -> Linear(1280 -> num_classes)
  The classifier is intentionally a single linear layer (multinomial
  logistic-regression head), with NO dropout and NO hidden layer.

Training:
  - 8-epoch warm-up
  - up to 32-epoch hard-example cascade
  - class-weighted CrossEntropyLoss
  - weighted sampler
  - difficulty-aware sampling during cascade
  - Adam
  - ReduceLROnPlateau on validation loss
  - early stopping on validation macro-F1

Extended evaluation:
  - train / val / test classification reports
  - train / val / test confusion matrices
  - train / val / test per-class accuracy
  - macro accuracy / precision / recall / F1
  - QWK + chi-square p-value (reported as a legacy diagnostic)
  - ROC curves + macro ROC-AUC
  - calibration curve + ECE
  - confidence histogram
  - per-class F1 heatmap
  - Grad-CAM
  - misclassified grid
  - model parameters / disk size / FLOPs
  - architecture summary
  - defense summary

NOTE FOR SCIENTIFIC REPORTING:
  The six disease classes are nominal categories, so QWK is retained only
  as an auxiliary/legacy diagnostic because it was present in the original
  pipeline. It should NOT be presented as the primary statistical metric.
=================================================================================
"""

# =============================================================================
# 0. CUDA allocator MUST be configured before importing torch
# =============================================================================
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

# =============================================================================
# 1. Imports
# =============================================================================
import gc
import re
import json
import pickle
import random
import time
import copy
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler

import torchvision
from torchvision import transforms, models

from PIL import Image

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from scipy.stats import chi2_contingency

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    f1_score,
    precision_recall_fscore_support,
    accuracy_score,
    cohen_kappa_score,
    balanced_accuracy_score,
)
from sklearn.preprocessing import label_binarize

import h5py

warnings.filterwarnings("ignore")

# =============================================================================
# 2. CONFIGURATION
# =============================================================================

DATASET_ROOT = Path(
    r"C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN"
)

OUTPUT_DIR = Path(
    r"C:\Users\R Y Z E N\Desktop\m4_coordatt_logreg_RTX2050_HARDENED"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "data_dir": str(DATASET_ROOT),
    "output_dir": str(OUTPUT_DIR),

    "img_size": 224,

    # RTX 2050 4 GB VRAM safe configuration
    "batch_size": 16,
    "gradient_accumulation_steps": 2,
    "effective_batch_size": 32,
    "num_workers": 0,
    "pin_memory": True,

    # Original M4 schedule
    "warmup_epochs": 8,
    "cascade_epochs": 32,
    "total_epochs": 40,

    "learning_rate": 1e-4,
    "weight_decay": 1e-4,

    "seed": 42,

    "early_stop_patience": 10,
    "lr_patience": 3,
    "lr_factor": 0.5,
    "min_lr": 1e-7,

    "ca_reduction": 32,
    "difficulty_alpha": 1.5,

    "gradcam_per_class": 2,
    "misclassified_grid_max": 16,

    # RTX 2050 hardening
    "use_amp": True,
    "checkpoint_every_epoch": True,

    # M4 imbalance strategy
    "use_class_weights": True,
    "use_weighted_sampler": True,
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# =============================================================================
# 3. Logging / GPU diagnostics
# =============================================================================

def log(msg):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] {msg}")


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        except Exception:
            pass


def print_gpu_memory(prefix=""):
    if not torch.cuda.is_available():
        return

    try:
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        free_mem, total_mem = torch.cuda.mem_get_info()
        free_gb = free_mem / 1e9

        print(
            f"{prefix}"
            f"GPU allocated={allocated:.2f} GB | "
            f"reserved={reserved:.2f} GB | "
            f"free={free_gb:.2f} GB / "
            f"{total_mem / 1e9:.2f} GB"
        )
    except Exception:
        pass


# =============================================================================
# 4. Reproducibility
# =============================================================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": (
            torch.get_rng_state()
            .detach()
            .cpu()
            .to(torch.uint8)
            .contiguous()
        ),
    }

    if torch.cuda.is_available():
        state["torch_cuda"] = [
            s.detach().cpu().to(torch.uint8).contiguous()
            for s in torch.cuda.get_rng_state_all()
        ]

    return state


def set_rng_state(state):
    if "python" in state:
        random.setstate(state["python"])

    if "numpy" in state:
        np.random.set_state(state["numpy"])

    if "torch" in state:
        torch_state = state["torch"]

        if isinstance(torch_state, torch.Tensor):
            torch_state = (
                torch_state.detach()
                .cpu()
                .to(torch.uint8)
                .contiguous()
            )
        else:
            torch_state = torch.tensor(
                torch_state,
                dtype=torch.uint8,
                device="cpu",
            ).contiguous()

        torch.set_rng_state(torch_state)

    if torch.cuda.is_available() and "torch_cuda" in state:
        fixed_cuda_states = []

        for cuda_state in state["torch_cuda"]:
            if isinstance(cuda_state, torch.Tensor):
                cuda_state = (
                    cuda_state.detach()
                    .cpu()
                    .to(torch.uint8)
                    .contiguous()
                )
            else:
                cuda_state = torch.tensor(
                    cuda_state,
                    dtype=torch.uint8,
                    device="cpu",
                ).contiguous()

            fixed_cuda_states.append(cuda_state)

        if fixed_cuda_states:
            torch.cuda.set_rng_state_all(fixed_cuda_states)


# =============================================================================
# 5. Dataset discovery
# =============================================================================

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"\nDataset directory not found:\n{DATASET_ROOT}\n"
        "Check DATASET_ROOT at the top of the script."
    )


def find_split_directory(root, names):
    """Find train/val/test directory recursively."""
    if isinstance(names, str):
        names = [names]

    candidates = []

    for p in root.rglob("*"):
        if not p.is_dir():
            continue

        if p.name.lower() in {x.lower() for x in names}:
            class_dirs = [
                x for x in p.iterdir()
                if x.is_dir()
            ]

            if len(class_dirs) >= 2:
                candidates.append((len(class_dirs), p))

    if not candidates:
        return None

    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1]


TRAIN_DIR = find_split_directory(DATASET_ROOT, "train")
VAL_DIR = find_split_directory(DATASET_ROOT, ["val", "validation"])
TEST_DIR = find_split_directory(DATASET_ROOT, "test")

if TRAIN_DIR is None:
    raise FileNotFoundError(
        f"Could not find a TRAIN directory under:\n{DATASET_ROOT}"
    )

if VAL_DIR is None:
    raise FileNotFoundError(
        f"Could not find a VAL/VALIDATION directory under:\n{DATASET_ROOT}"
    )

if TEST_DIR is None:
    raise FileNotFoundError(
        f"Could not find a TEST directory under:\n{DATASET_ROOT}"
    )

log("=" * 70)
log("ORALVISION-M4 — RTX 2050 HARDENED VERSION")
log("=" * 70)
log(f"PyTorch version : {torch.__version__}")
log(f"Torchvision     : {torchvision.__version__}")
log(f"Device          : {DEVICE}")
log(f"Dataset root    : {DATASET_ROOT}")
log(f"Output directory: {OUTPUT_DIR}")
log(f"Train directory : {TRAIN_DIR}")
log(f"Val directory   : {VAL_DIR}")
log(f"Test directory  : {TEST_DIR}")
log(f"Real batch size : {CONFIG['batch_size']}")
log(f"Accumulation    : x{CONFIG['gradient_accumulation_steps']}")
log(f"Effective batch : {CONFIG['effective_batch_size']}")
log(f"AMP enabled     : {CONFIG['use_amp']}")

if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_TOTAL = (
        torch.cuda.get_device_properties(0).total_memory / 1e9
    )
    log(f"GPU             : {GPU_NAME}")
    log(f"GPU memory      : {GPU_TOTAL:.2f} GB")
    print_gpu_memory("Initial GPU memory: ")

    if GPU_TOTAL <= 5.0:
        log("RTX-class low-VRAM mode detected.")
else:
    log("WARNING: CUDA is not available. Training will run on CPU.")

log("=" * 70)

# =============================================================================
# 6. Dataset
# =============================================================================

VALID_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}


def discover_classes(train_dir, val_dir, test_dir):
    train_classes = {
        p.name for p in train_dir.iterdir() if p.is_dir()
    }
    val_classes = {
        p.name for p in val_dir.iterdir() if p.is_dir()
    }
    test_classes = {
        p.name for p in test_dir.iterdir() if p.is_dir()
    }

    if train_classes != val_classes or train_classes != test_classes:
        raise RuntimeError(
            "\nClass directories do not match across train/val/test.\n"
            f"Train: {sorted(train_classes)}\n"
            f"Val:   {sorted(val_classes)}\n"
            f"Test:  {sorted(test_classes)}"
        )

    return sorted(train_classes)


CLASS_NAMES = discover_classes(TRAIN_DIR, VAL_DIR, TEST_DIR)
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: name for name, i in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASS_NAMES)


def collect_samples(split_dir):
    samples = []

    for class_name in CLASS_NAMES:
        class_dir = split_dir / class_name
        for path in sorted(class_dir.rglob("*")):
            if path.is_file() and path.suffix.lower() in VALID_EXTENSIONS:
                samples.append(
                    (str(path), CLASS_TO_IDX[class_name])
                )

    return samples


TRAIN_SAMPLES = collect_samples(TRAIN_DIR)
VAL_SAMPLES = collect_samples(VAL_DIR)
TEST_SAMPLES = collect_samples(TEST_DIR)

if not TRAIN_SAMPLES or not VAL_SAMPLES or not TEST_SAMPLES:
    raise RuntimeError(
        "One or more dataset splits contain zero images."
    )


class OralDataset(Dataset):
    """
    Returns image, label, global index.

    Global indices are needed by the difficulty-aware cascade sampler.
    """

    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]

        with Image.open(path) as img:
            img = img.convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        return img, label, index


# =============================================================================
# 7. Transforms
# =============================================================================

train_transform = transforms.Compose([
    transforms.Resize(
        (CONFIG["img_size"], CONFIG["img_size"])
    ),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(
        (CONFIG["img_size"], CONFIG["img_size"])
    ),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dataset = OralDataset(
    TRAIN_SAMPLES,
    transform=train_transform,
)

val_dataset = OralDataset(
    VAL_SAMPLES,
    transform=eval_transform,
)

test_dataset = OralDataset(
    TEST_SAMPLES,
    transform=eval_transform,
)


def make_eval_loader(dataset):
    return DataLoader(
        dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=(
            CONFIG["pin_memory"]
            and DEVICE.type == "cuda"
        ),
    )


val_loader = make_eval_loader(val_dataset)
test_loader = make_eval_loader(test_dataset)

log(
    f"Dataset sizes -> "
    f"train={len(TRAIN_SAMPLES)}, "
    f"val={len(VAL_SAMPLES)}, "
    f"test={len(TEST_SAMPLES)}"
)
log(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

# =============================================================================
# 8. Class weights
# =============================================================================

train_labels = np.array(
    [label for _, label in TRAIN_SAMPLES],
    dtype=np.int64,
)

if CONFIG["use_class_weights"]:
    class_weight_values = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(NUM_CLASSES),
        y=train_labels,
    )

    class_weights = {
        int(i): float(w)
        for i, w in enumerate(class_weight_values)
    }
else:
    class_weights = {
        i: 1.0 for i in range(NUM_CLASSES)
    }

log(
    "Class weights: "
    + str({
        IDX_TO_CLASS[i]: round(class_weights[i], 4)
        for i in range(NUM_CLASSES)
    })
)

class_weights_tensor = torch.tensor(
    [class_weights[i] for i in range(NUM_CLASSES)],
    dtype=torch.float32,
    device=DEVICE,
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)

# =============================================================================
# 9. Coordinate Attention
# =============================================================================

class CoordAtt(nn.Module):
    """
    Coordinate Attention.

    The spatial feature map is pooled independently along H and W so that
    positional information is retained before generating horizontal and
    vertical attention maps.
    """

    def __init__(self, channels, reduction=32):
        super().__init__()

        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))

        mip = max(8, channels // reduction)

        self.conv1 = nn.Conv2d(
            channels,
            mip,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.Hardswish(inplace=True)

        self.conv_h = nn.Conv2d(
            mip,
            channels,
            kernel_size=1,
            stride=1,
            padding=0,
        )

        self.conv_w = nn.Conv2d(
            mip,
            channels,
            kernel_size=1,
            stride=1,
            padding=0,
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        identity = x
        _, _, h, w = x.size()

        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)

        y = torch.cat(
            [x_h, x_w],
            dim=2,
        )

        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)

        x_h, x_w = torch.split(
            y,
            [h, w],
            dim=2,
        )

        x_w = x_w.permute(
            0, 1, 3, 2
        )

        a_h = self.sigmoid(
            self.conv_h(x_h)
        )

        a_w = self.sigmoid(
            self.conv_w(x_w)
        )

        return identity * a_h * a_w


# =============================================================================
# 10. EfficientNet-B0 + CoordAtt + Logistic Regression
# =============================================================================

class EfficientNetCoordAtt(nn.Module):
    """
    EfficientNet-B0 + Coordinate Attention + pure logistic-regression head.

    IMPORTANT:
      The classifier is exactly:
          Linear(1280 -> num_classes)

      There is:
        - no dropout
        - no hidden layer
        - no extra MLP
    """

    def __init__(
        self,
        num_classes,
        ca_reduction=32,
        pretrained=True,
    ):
        super().__init__()

        weights = (
            models.EfficientNet_B0_Weights.DEFAULT
            if pretrained
            else None
        )

        base = models.efficientnet_b0(
            weights=weights
        )

        self.features = base.features

        self.coordatt = CoordAtt(
            1280,
            reduction=ca_reduction,
        )

        self.avgpool = nn.AdaptiveAvgPool2d(1)

        # PURE LOGISTIC REGRESSION HEAD
        self.classifier = nn.Linear(
            1280,
            num_classes,
        )

    def forward(self, x):
        x = self.features(x)
        x = self.coordatt(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        return self.classifier(x)


# =============================================================================
# 11. Sampler / difficulty tracking
# =============================================================================

difficulty_dict = {}


def build_train_loader():
    if not CONFIG["use_weighted_sampler"]:
        return DataLoader(
            train_dataset,
            batch_size=CONFIG["batch_size"],
            shuffle=True,
            num_workers=CONFIG["num_workers"],
            pin_memory=(
                CONFIG["pin_memory"]
                and DEVICE.type == "cuda"
            ),
        )

    weights = []

    for i, (_, label) in enumerate(TRAIN_SAMPLES):
        weight = class_weights[label]

        difficulty = difficulty_dict.get(
            i,
            1.0,
        )

        # Prevent pathological sampler weights
        difficulty = max(
            0.25,
            min(
                float(difficulty),
                10.0,
            ),
        )

        # During the first epochs, difficulty_dict is near 1.
        # During cascade it progressively emphasizes hard examples.
        weight *= (
            difficulty
            ** CONFIG["difficulty_alpha"]
        )

        weights.append(weight)

    weights = np.asarray(
        weights,
        dtype=np.float64,
    )

    weights = np.maximum(
        weights,
        1e-12,
    )

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(
            weights,
            dtype=torch.double,
        ),
        num_samples=len(weights),
        replacement=True,
    )

    return DataLoader(
        train_dataset,
        batch_size=CONFIG["batch_size"],
        sampler=sampler,
        num_workers=CONFIG["num_workers"],
        pin_memory=(
            CONFIG["pin_memory"]
            and DEVICE.type == "cuda"
        ),
    )


# =============================================================================
# 12. Metrics
# =============================================================================

def calculate_metrics(labels, preds):
    labels = np.asarray(labels)
    preds = np.asarray(preds)

    return {
        "accuracy": float(
            accuracy_score(labels, preds)
        ),
        "macro_f1": float(
            f1_score(
                labels,
                preds,
                average="macro",
                zero_division=0,
            )
        ),
        "weighted_f1": float(
            f1_score(
                labels,
                preds,
                average="weighted",
                zero_division=0,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                labels,
                preds,
            )
        ),
    }


# =============================================================================
# 13. Train / validation loop
# =============================================================================

def run_train_epoch(
    model,
    loader,
    optimizer,
    scaler,
    epoch,
):
    model.train()

    running_loss = 0.0
    total_samples = 0

    epoch_labels = []
    epoch_preds = []

    optimizer.zero_grad(
        set_to_none=True
    )

    accum_steps = CONFIG[
        "gradient_accumulation_steps"
    ]

    for step, (
        images,
        labels,
        indices,
    ) in enumerate(loader):

        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True,
        )

        is_step = (
            (step + 1) % accum_steps == 0
            or (step + 1) == len(loader)
        )

        outputs = None
        per_sample_loss = None
        raw_loss = None
        loss = None

        try:
            with autocast(
                device_type="cuda",
                enabled=(
                    CONFIG["use_amp"]
                    and DEVICE.type == "cuda"
                ),
            ):
                outputs = model(images)

                per_sample_loss = (
                    nn.functional.cross_entropy(
                        outputs,
                        labels,
                        weight=criterion.weight,
                        reduction="none",
                    )
                )

                raw_loss = per_sample_loss.mean()

                loss = raw_loss / accum_steps

            scaler.scale(loss).backward()

        except torch.cuda.OutOfMemoryError:
            log(
                f"\n!! CUDA OOM at epoch={epoch}, "
                f"batch={step + 1}/{len(loader)}"
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            del outputs
            del per_sample_loss
            del raw_loss
            del loss
            del images
            del labels

            cleanup_gpu()
            print_gpu_memory(
                "After OOM cleanup: "
            )

            log(
                "The current batch could not fit "
                "in available VRAM."
            )
            log(
                "The checkpoint from the last "
                "completed epoch remains intact."
            )
            log(
                "If OOM repeats, use "
                "batch_size=8 and "
                "gradient_accumulation_steps=4."
            )

            raise

        # Update difficulty using the UNDIVIDED per-sample loss
        with torch.no_grad():
            losses_np = (
                per_sample_loss
                .detach()
                .float()
                .cpu()
                .numpy()
            )

            for gi, sample_loss in zip(
                indices.numpy(),
                losses_np,
            ):
                gi = int(gi)
                previous = difficulty_dict.get(
                    gi,
                    1.0,
                )

                # Exponential moving average
                difficulty_dict[gi] = (
                    0.5 * previous
                    + 0.5 * float(sample_loss)
                )

            predictions = torch.argmax(
                outputs,
                dim=1,
            )

            epoch_labels.extend(
                labels.detach()
                .cpu()
                .numpy()
                .tolist()
            )

            epoch_preds.extend(
                predictions.detach()
                .cpu()
                .numpy()
                .tolist()
            )

        batch_size = images.size(0)

        running_loss += (
            raw_loss.item()
            * batch_size
        )

        total_samples += batch_size

        if is_step:
            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(
                set_to_none=True
            )

        del images
        del labels
        del outputs
        del per_sample_loss
        del raw_loss
        del loss
        del predictions

    metrics = calculate_metrics(
        epoch_labels,
        epoch_preds,
    )

    metrics["loss"] = (
        running_loss / max(total_samples, 1)
    )

    return metrics


@torch.no_grad()
def evaluate_loader(
    model,
    loader,
):
    model.eval()

    running_loss = 0.0
    total_samples = 0

    all_labels = []
    all_preds = []
    all_probs = []

    for images, labels, _ in loader:
        images = images.to(
            DEVICE,
            non_blocking=True,
        )

        labels_device = labels.to(
            DEVICE,
            non_blocking=True,
        )

        with autocast(
            device_type="cuda",
            enabled=(
                CONFIG["use_amp"]
                and DEVICE.type == "cuda"
            ),
        ):
            outputs = model(images)

            loss = criterion(
                outputs,
                labels_device,
            )

        probs = torch.softmax(
            outputs,
            dim=1,
        )

        preds = torch.argmax(
            outputs,
            dim=1,
        )

        batch_size = images.size(0)

        running_loss += (
            loss.item()
            * batch_size
        )

        total_samples += batch_size

        all_labels.extend(
            labels.numpy().tolist()
        )

        all_preds.extend(
            preds.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        all_probs.extend(
            probs.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        del images
        del labels_device
        del outputs
        del loss
        del probs
        del preds

    metrics = calculate_metrics(
        all_labels,
        all_preds,
    )

    metrics["loss"] = (
        running_loss / max(total_samples, 1)
    )

    return (
        metrics,
        np.asarray(all_labels),
        np.asarray(all_preds),
        np.asarray(all_probs),
    )


# =============================================================================
# 14. Checkpoint helpers
# =============================================================================

CHECKPOINT_PATH = (
    OUTPUT_DIR / "checkpoint.pth"
)


def save_checkpoint(
    epoch,
    model,
    optimizer,
    scheduler,
    scaler,
    history,
    best_val_f1,
    best_epoch,
    best_state,
    patience_counter,
    total_training_time,
):
    checkpoint = {
        "checkpoint_version": 3,
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),

        "history": history,

        "best_val_macro_f1": best_val_f1,
        "best_epoch": best_epoch,
        "best_state": best_state,
        "patience_counter": patience_counter,

        "difficulty_dict": difficulty_dict,

        "rng_state": get_rng_state(),

        "config": CONFIG,
        "class_names": CLASS_NAMES,

        "total_training_time": total_training_time,
    }

    temp_path = (
        OUTPUT_DIR / "checkpoint.tmp"
    )

    torch.save(
        checkpoint,
        temp_path,
    )

    # Atomic replacement
    temp_path.replace(
        CHECKPOINT_PATH
    )


def load_checkpoint():
    if not CHECKPOINT_PATH.exists():
        return None

    log(
        f"Loading checkpoint:\n"
        f"{CHECKPOINT_PATH}"
    )

    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    required = [
        "epoch",
        "model_state_dict",
        "optimizer_state_dict",
        "scheduler_state_dict",
        "scaler_state_dict",
        "history",
        "best_val_macro_f1",
        "best_epoch",
        "best_state",
        "patience_counter",
        "rng_state",
    ]

    missing = [
        k for k in required
        if k not in checkpoint
    ]

    if missing:
        raise RuntimeError(
            f"Checkpoint is missing keys: {missing}"
        )

    return checkpoint


# =============================================================================
# 15. Grad-CAM
# =============================================================================

class GradCAM:
    def __init__(
        self,
        model,
        target_layer,
    ):
        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        self.forward_handle = (
            target_layer.register_forward_hook(
                self._save_activation
            )
        )

        self.backward_handle = (
            target_layer.register_full_backward_hook(
                self._save_gradient
            )
        )

    def _save_activation(
        self,
        module,
        inputs,
        output,
    ):
        self.activations = output.detach()

    def _save_gradient(
        self,
        module,
        grad_input,
        grad_output,
    ):
        self.gradients = (
            grad_output[0].detach()
        )

    def generate(
        self,
        input_tensor,
        class_idx=None,
    ):
        self.model.eval()

        self.model.zero_grad(
            set_to_none=True
        )

        output = self.model(
            input_tensor
        )

        if class_idx is None:
            class_idx = int(
                output.argmax(
                    dim=1
                ).item()
            )

        score = output[:, class_idx]

        score.backward(
            retain_graph=True
        )

        gradients = self.gradients[0]
        activations = self.activations[0]

        weights = gradients.mean(
            dim=(1, 2)
        )

        cam = torch.zeros(
            activations.shape[1:],
            dtype=torch.float32,
            device=activations.device,
        )

        for channel, weight in enumerate(
            weights
        ):
            cam += (
                weight
                * activations[channel]
            )

        cam = torch.relu(cam)

        cam -= cam.min()

        cam = cam / (
            cam.max() + 1e-8
        )

        confidence = torch.softmax(
            output,
            dim=1,
        )[0, class_idx].item()

        return (
            cam.detach().cpu().numpy(),
            class_idx,
            confidence,
        )

    def remove_hooks(self):
        self.forward_handle.remove()
        self.backward_handle.remove()


def overlay_heatmap_on_image(
    image,
    cam,
    alpha=0.45,
):
    width, height = image.size

    cam_uint8 = (
        cam * 255
    ).astype(np.uint8)

    cam_resized = (
        np.asarray(
            Image.fromarray(
                cam_uint8
            ).resize(
                (width, height),
                Image.BILINEAR,
            )
        )
        / 255.0
    )

    heatmap_rgb = (
        cm.jet(
            cam_resized
        )[:, :, :3]
        * 255
    ).astype(np.uint8)

    original = np.asarray(
        image
    ).astype(np.uint8)

    overlay = (
        heatmap_rgb.astype(np.float32)
        * alpha
        + original.astype(np.float32)
        * (1 - alpha)
    ).astype(np.uint8)

    return heatmap_rgb, overlay


def run_gradcam_analysis(
    model,
    samples,
    indices_by_class,
):
    gradcam_dir = (
        OUTPUT_DIR / "gradcam"
    )

    gradcam_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    target_layer = model.coordatt
    gradcam = GradCAM(
        model,
        target_layer,
    )

    eval_tf = transforms.Compose([
        transforms.Resize(
            (
                CONFIG["img_size"],
                CONFIG["img_size"],
            )
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            IMAGENET_MEAN,
            IMAGENET_STD,
        ),
    ])

    grid_rows = []

    try:
        for class_idx, index_list in (
            indices_by_class.items()
        ):
            class_name = IDX_TO_CLASS[
                class_idx
            ]

            selected = index_list[
                :CONFIG["gradcam_per_class"]
            ]

            for n, index in enumerate(
                selected
            ):
                path, true_label = (
                    samples[index]
                )

                raw_img = (
                    Image.open(path)
                    .convert("RGB")
                    .resize(
                        (
                            CONFIG["img_size"],
                            CONFIG["img_size"],
                        )
                    )
                )

                input_tensor = eval_tf(
                    Image.open(path)
                    .convert("RGB")
                ).unsqueeze(0).to(
                    DEVICE
                )

                cam, pred_idx, confidence = (
                    gradcam.generate(
                        input_tensor
                    )
                )

                heatmap, overlay = (
                    overlay_heatmap_on_image(
                        raw_img,
                        cam,
                    )
                )

                correct = (
                    "correct"
                    if pred_idx == true_label
                    else "WRONG"
                )

                fig, axes = plt.subplots(
                    1,
                    3,
                    figsize=(9, 3.3),
                )

                axes[0].imshow(raw_img)
                axes[0].set_title(
                    "Original"
                )
                axes[0].axis("off")

                axes[1].imshow(heatmap)
                axes[1].set_title(
                    "Grad-CAM"
                )
                axes[1].axis("off")

                axes[2].imshow(overlay)
                axes[2].set_title(
                    "Overlay"
                )
                axes[2].axis("off")

                fig.suptitle(
                    f"{class_name} | "
                    f"pred: {IDX_TO_CLASS[pred_idx]} "
                    f"({confidence:.2f}) "
                    f"[{correct}]",
                    fontsize=10,
                )

                plt.tight_layout()

                plt.savefig(
                    gradcam_dir
                    / f"gradcam_{class_name}_{n}.png",
                    dpi=130,
                )

                plt.close(fig)

                grid_rows.append(
                    (
                        class_name,
                        raw_img,
                        heatmap,
                        overlay,
                        IDX_TO_CLASS[pred_idx],
                        confidence,
                    )
                )

                del input_tensor
                cleanup_gpu()

        if grid_rows:
            n_rows = len(grid_rows)

            fig, axes = plt.subplots(
                n_rows,
                3,
                figsize=(
                    9,
                    3 * n_rows,
                ),
            )

            if n_rows == 1:
                axes = axes.reshape(
                    1,
                    3,
                )

            for r, row in enumerate(
                grid_rows
            ):
                class_name, raw_img, heatmap, overlay, pred_name, confidence = row

                axes[r, 0].imshow(
                    raw_img
                )
                axes[r, 0].axis("off")
                axes[r, 0].set_ylabel(
                    class_name,
                    fontsize=9,
                )

                axes[r, 1].imshow(
                    heatmap
                )
                axes[r, 1].axis("off")

                axes[r, 2].imshow(
                    overlay
                )
                axes[r, 2].axis("off")

                if r == 0:
                    axes[r, 0].set_title(
                        "Original"
                    )
                    axes[r, 1].set_title(
                        "Grad-CAM"
                    )

                axes[r, 2].set_title(
                    f"pred: {pred_name} "
                    f"({confidence:.2f})",
                    fontsize=8,
                )

            plt.tight_layout()

            plt.savefig(
                OUTPUT_DIR
                / "gradcam_grid.png",
                dpi=130,
            )

            plt.close(fig)

    finally:
        gradcam.remove_hooks()
        cleanup_gpu()

    log(
        f"Grad-CAM complete: "
        f"{len(grid_rows)} overlays saved."
    )


# =============================================================================
# 16. Plot helpers
# =============================================================================

def per_class_accuracy_from_cm(cm_arr):
    row_sums = cm_arr.sum(axis=1)

    values = np.divide(
        np.diag(cm_arr),
        row_sums,
        out=np.zeros(
            len(row_sums),
            dtype=float,
        ),
        where=row_sums != 0,
    )

    return values, row_sums


def plot_per_class_accuracy(
    cm_arr,
    class_names,
    split_name=None,
):
    values, supports = (
        per_class_accuracy_from_cm(
            cm_arr
        )
    )

    prefix = (
        f"{split_name}_"
        if split_name
        else ""
    )

    title = (
        split_name.capitalize()
        if split_name
        else "Test"
    )

    plt.figure(
        figsize=(8, 5)
    )

    bars = plt.bar(
        class_names,
        values,
    )

    for bar, value, support in zip(
        bars,
        values,
        supports,
    ):
        plt.text(
            bar.get_x()
            + bar.get_width() / 2,
            min(
                value + 0.01,
                1.08,
            ),
            f"{value:.3f}\n(n={int(support)})",
            ha="center",
            fontsize=8,
        )

    plt.ylim(
        0,
        1.15,
    )

    plt.ylabel(
        "Per-class Accuracy (Recall)"
    )

    plt.title(
        f"Per-class Accuracy - {title} Set"
    )

    plt.xticks(
        rotation=25,
        ha="right",
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / f"{prefix}per_class_accuracy.png",
        dpi=150,
    )

    plt.close()


def save_per_class_accuracy_txt(
    cm_arr,
    class_names,
    split_name=None,
):
    values, supports = (
        per_class_accuracy_from_cm(
            cm_arr
        )
    )

    correct = np.diag(
        cm_arr
    )

    prefix = (
        f"{split_name}_"
        if split_name
        else ""
    )

    title = (
        split_name.capitalize()
        if split_name
        else "Test"
    )

    lines = [
        f"OralVision - Per-Class Accuracy ({title} Set)",
        "=" * 60,
        f"{'Class':22s} {'Accuracy':>10s} "
        f"{'Correct':>10s} {'Total':>8s}",
        "-" * 60,
    ]

    for name, value, c, total in zip(
        class_names,
        values,
        correct,
        supports,
    ):
        lines.append(
            f"{name:22s} "
            f"{value:10.4f} "
            f"{int(c):10d} "
            f"{int(total):8d}"
        )

    overall = (
        correct.sum()
        / max(
            supports.sum(),
            1,
        )
    )

    lines.extend([
        "-" * 60,
        f"{'OVERALL':22s} "
        f"{overall:10.4f} "
        f"{int(correct.sum()):10d} "
        f"{int(supports.sum()):8d}",
    ])

    with open(
        OUTPUT_DIR
        / f"{prefix}per_class_accuracy.txt",
        "w",
        encoding="utf-8",
    ) as f:
        f.write(
            "\n".join(lines)
        )


def save_architecture_summary(model):
    total_params = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    lines = [
        "OralVision - Model Architecture Summary",
        "=" * 60,
        f"Total parameters:     {total_params:,}",
        f"Trainable parameters: {trainable_params:,}",
        "",
        "Architecture:",
        "EfficientNet-B0 -> Coordinate Attention -> GAP -> Linear",
        "",
        "Classifier head:",
        "Linear(1280 -> num_classes)",
        "No dropout",
        "No hidden layer",
        "",
        "Full layer-by-layer structure:",
        "-" * 60,
        str(model),
    ]

    with open(
        OUTPUT_DIR
        / "model_architecture.txt",
        "w",
        encoding="utf-8",
    ) as f:
        f.write(
            "\n".join(lines)
        )


def plot_misclassified_grid(
    samples,
    labels,
    preds,
):
    wrong = [
        (
            i,
            true_label,
            pred_label,
        )
        for i, (
            true_label,
            pred_label,
        ) in enumerate(
            zip(labels, preds)
        )
        if true_label != pred_label
    ]

    if not wrong:
        log(
            "No misclassified test samples."
        )
        return

    wrong = wrong[
        :CONFIG[
            "misclassified_grid_max"
        ]
    ]

    cols = 4
    rows = (
        len(wrong) + cols - 1
    ) // cols

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(
            cols * 2.6,
            rows * 2.8,
        ),
    )

    axes = np.asarray(
        axes
    ).reshape(-1)

    for i, (
        global_index,
        true_idx,
        pred_idx,
    ) in enumerate(wrong):

        path, _ = samples[
            global_index
        ]

        image = (
            Image.open(path)
            .convert("RGB")
            .resize(
                (
                    CONFIG["img_size"],
                    CONFIG["img_size"],
                )
            )
        )

        axes[i].imshow(
            image
        )

        axes[i].set_title(
            f"true: {IDX_TO_CLASS[true_idx]}\n"
            f"pred: {IDX_TO_CLASS[pred_idx]}",
            fontsize=8,
        )

        axes[i].axis("off")

    for j in range(
        len(wrong),
        len(axes),
    ):
        axes[j].axis("off")

    plt.suptitle(
        f"Misclassified test samples "
        f"({len(wrong)} shown)"
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / "misclassified_grid.png",
        dpi=130,
    )

    plt.close(fig)


# =============================================================================
# 17. Extended evaluation
# =============================================================================

def qwk_and_chi2(
    labels,
    preds,
):
    cm_arr = confusion_matrix(
        labels,
        preds,
        labels=list(
            range(NUM_CLASSES)
        ),
    )

    # QWK is retained for continuity with the original pipeline.
    try:
        qwk = cohen_kappa_score(
            labels,
            preds,
            weights="quadratic",
        )
    except Exception:
        qwk = float("nan")

    cm_safe = (
        cm_arr.astype(float)
        + 1e-9
    )

    try:
        chi2_stat, p_value, dof, _ = (
            chi2_contingency(
                cm_safe
            )
        )
    except Exception:
        chi2_stat = float("nan")
        p_value = float("nan")
        dof = 0

    return (
        qwk,
        chi2_stat,
        dof,
        p_value,
        cm_arr,
    )


def compute_macro_auc(
    labels,
    probs,
):
    y_true_bin = label_binarize(
        labels,
        classes=list(
            range(NUM_CLASSES)
        ),
    )

    aucs = []

    for c in range(
        NUM_CLASSES
    ):
        try:
            fpr, tpr, _ = roc_curve(
                y_true_bin[:, c],
                probs[:, c],
            )

            aucs.append(
                auc(
                    fpr,
                    tpr,
                )
            )
        except Exception:
            aucs.append(
                np.nan
            )

    return float(
        np.nanmean(aucs)
    )


def plot_extended_metrics_bar(
    results
):
    splits = list(
        results.keys()
    )

    metric_keys = [
        "acc",
        "prec",
        "rec",
        "f1",
    ]

    metric_labels = [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
    ]

    x = np.arange(
        len(splits)
    )

    width = 0.20

    fig, ax = plt.subplots(
        figsize=(9, 5.5)
    )

    for i, (
        key,
        label,
    ) in enumerate(
        zip(
            metric_keys,
            metric_labels,
        )
    ):
        values = [
            results[s][key]
            for s in splits
        ]

        bars = ax.bar(
            x
            + (
                i - 1.5
            ) * width,
            values,
            width,
            label=label,
        )

        for bar, value in zip(
            bars,
            values,
        ):
            ax.text(
                bar.get_x()
                + bar.get_width()
                / 2,
                min(
                    value + 0.003,
                    1.10,
                ),
                f"{value:.3f}",
                ha="center",
                fontsize=7,
                rotation=90,
            )

    ax.set_xticks(x)

    ax.set_xticklabels(
        [
            s.capitalize()
            for s in splits
        ]
    )

    ax.set_ylim(
        0,
        1.15,
    )

    ax.set_ylabel(
        "Score"
    )

    ax.set_title(
        "Macro Metrics by Split"
    )

    ax.legend(
        loc="lower right",
        ncol=4,
        fontsize=8,
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / "extended_metrics_bar.png",
        dpi=150,
    )

    plt.close(fig)


def plot_qwk_pvalue_summary(
    results
):
    splits = list(
        results.keys()
    )

    qwks = [
        results[s]["qwk"]
        for s in splits
    ]

    pvals = [
        results[s]["p_value"]
        for s in splits
    ]

    neg_log_p = [
        -np.log10(
            max(
                p,
                1e-300,
            )
        )
        if np.isfinite(p)
        else 0.0
        for p in pvals
    ]

    fig, (
        ax1,
        ax2,
    ) = plt.subplots(
        1,
        2,
        figsize=(11, 5),
    )

    ax1.bar(
        range(len(splits)),
        qwks,
    )

    ax1.set_xticks(
        range(len(splits))
    )

    ax1.set_xticklabels(
        [
            s.capitalize()
            for s in splits
        ]
    )

    ax1.set_ylim(
        0,
        1.05,
    )

    ax1.set_ylabel(
        "Quadratic Weighted Kappa"
    )

    ax1.set_title(
        "QWK by Split"
    )

    for i, value in enumerate(
        qwks
    ):
        if np.isfinite(value):
            ax1.text(
                i,
                value + 0.02,
                f"{value:.4f}",
                ha="center",
            )

    ax2.bar(
        range(len(splits)),
        neg_log_p,
    )

    ax2.set_xticks(
        range(len(splits))
    )

    ax2.set_xticklabels(
        [
            s.capitalize()
            for s in splits
        ]
    )

    ax2.set_ylabel(
        "-log10(p-value)"
    )

    ax2.set_title(
        "Chi-square Diagnostic"
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / "qwk_pvalue_summary.png",
        dpi=150,
    )

    plt.close()


def plot_model_complexity_summary(
    total_params,
    trainable_params,
    size_mb,
    flops,
):
    fig, ax = plt.subplots(
        figsize=(7, 4.5)
    )

    ax.axis("off")

    if flops is None:
        flops_text = (
            "N/A (install thop)"
        )
    else:
        flops_text = (
            f"{flops / 1e9:.3f} GFLOPs "
            f"({flops:,.0f} FLOPs)"
        )

    rows = [
        (
            "Total parameters",
            f"{total_params:,}",
        ),
        (
            "Trainable parameters",
            f"{trainable_params:,}",
        ),
        (
            "Model size on disk",
            f"{size_mb:.2f} MB",
        ),
        (
            "FLOPs @ 224x224",
            flops_text,
        ),
    ]

    table = ax.table(
        cellText=[
            [k, v]
            for k, v in rows
        ],
        colLabels=[
            "Metric",
            "Value",
        ],
        cellLoc="left",
        colLoc="left",
        loc="center",
    )

    table.auto_set_font_size(
        False
    )

    table.set_fontsize(
        10
    )

    table.scale(
        1,
        2.2,
    )

    for (
        row,
        col,
    ), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(
                weight="bold"
            )

    ax.set_title(
        "Model Complexity Summary",
        fontsize=13,
        pad=20,
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / "model_complexity_summary.png",
        dpi=150,
    )

    plt.close(fig)


def plot_per_class_f1_heatmap(
    results
):
    matrix = np.array([
        results[s]["per_class_f1"]
        for s in results
    ])

    plt.figure(
        figsize=(
            max(
                8,
                NUM_CLASSES * 0.9,
            ),
            4.5,
        )
    )

    sns.heatmap(
        matrix,
        annot=True,
        fmt=".3f",
        cmap="RdYlGn",
        vmin=0,
        vmax=1,
        xticklabels=CLASS_NAMES,
        yticklabels=[
            s.capitalize()
            for s in results
        ],
        cbar_kws={
            "label": "F1-score"
        },
    )

    plt.title(
        "Per-Class F1 Across Splits"
    )

    plt.xticks(
        rotation=25,
        ha="right",
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / "per_class_f1_heatmap.png",
        dpi=150,
    )

    plt.close()


def plot_confidence_histogram(
    probs,
    labels,
    preds,
):
    confidences = probs.max(
        axis=1
    )

    correct_mask = (
        preds == labels
    )

    plt.figure(
        figsize=(8, 5)
    )

    plt.hist(
        confidences[
            correct_mask
        ],
        bins=20,
        range=(0, 1),
        alpha=0.6,
        label=(
            f"Correct "
            f"(n={correct_mask.sum()})"
        ),
    )

    plt.hist(
        confidences[
            ~correct_mask
        ],
        bins=20,
        range=(0, 1),
        alpha=0.6,
        label=(
            f"Incorrect "
            f"(n={(~correct_mask).sum()})"
        ),
    )

    plt.xlabel(
        "Maximum softmax confidence"
    )

    plt.ylabel(
        "Count"
    )

    plt.title(
        "Prediction Confidence - Test Set"
    )

    plt.legend()

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / "confidence_histogram_test.png",
        dpi=150,
    )

    plt.close()


def plot_calibration_curve(
    probs,
    labels,
    preds,
    n_bins=10,
):
    confidences = probs.max(
        axis=1
    )

    correct = (
        preds == labels
    ).astype(float)

    edges = np.linspace(
        0,
        1,
        n_bins + 1,
    )

    bin_conf = []
    bin_acc = []
    bin_counts = []

    for i in range(
        n_bins
    ):
        lo = edges[i]
        hi = edges[i + 1]

        if i < n_bins - 1:
            mask = (
                (confidences >= lo)
                & (confidences < hi)
            )
        else:
            mask = (
                (confidences >= lo)
                & (confidences <= hi)
            )

        if mask.sum() > 0:
            bin_conf.append(
                confidences[
                    mask
                ].mean()
            )

            bin_acc.append(
                correct[
                    mask
                ].mean()
            )

            bin_counts.append(
                int(mask.sum())
            )
        else:
            bin_conf.append(
                (lo + hi) / 2
            )

            bin_acc.append(
                np.nan
            )

            bin_counts.append(
                0
            )

    ece = (
        np.nansum([
            abs(c - a) * n
            for c, a, n in zip(
                bin_conf,
                bin_acc,
                bin_counts,
            )
            if n > 0
        ])
        / max(
            sum(bin_counts),
            1,
        )
    )

    plt.figure(
        figsize=(6.5, 6)
    )

    plt.plot(
        [0, 1],
        [0, 1],
        "k--",
        label="Perfect calibration",
    )

    plt.plot(
        bin_conf,
        bin_acc,
        "o-",
        label="Model",
    )

    plt.xlabel(
        "Mean predicted confidence"
    )

    plt.ylabel(
        "Empirical accuracy"
    )

    plt.title(
        f"Calibration Curve - Test Set\nECE = {ece:.4f}"
    )

    plt.legend()

    plt.xlim(
        0,
        1,
    )

    plt.ylim(
        0,
        1,
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / "calibration_curve_test.png",
        dpi=150,
    )

    plt.close()

    return float(ece)


def plot_macro_roc_auc_by_split(
    results
):
    splits = list(
        results.keys()
    )

    aucs = [
        results[s]["macro_auc"]
        for s in splits
    ]

    plt.figure(
        figsize=(7, 5)
    )

    bars = plt.bar(
        range(len(splits)),
        aucs,
    )

    for bar, value in zip(
        bars,
        aucs,
    ):
        if np.isfinite(value):
            plt.text(
                bar.get_x()
                + bar.get_width() / 2,
                value + 0.005,
                f"{value:.4f}",
                ha="center",
                fontsize=9,
            )

    plt.xticks(
        range(len(splits)),
        [
            s.capitalize()
            for s in splits
        ],
    )

    plt.ylim(
        0,
        1.05,
    )

    plt.ylabel(
        "Macro ROC-AUC (OvR)"
    )

    plt.title(
        "Macro ROC-AUC by Split"
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / "macro_roc_auc_by_split.png",
        dpi=150,
    )

    plt.close()


def compute_model_complexity(
    model,
    best_model_path,
):
    total_params = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    size_mb = (
        best_model_path.stat().st_size
        / (1024 ** 2)
        if best_model_path.exists()
        else float("nan")
    )

    flops = None

    try:
        from thop import profile

        model_was_training = model.training
        model.eval()

        dummy = torch.randn(
            1,
            3,
            CONFIG["img_size"],
            CONFIG["img_size"],
            device=DEVICE,
        )

        with torch.no_grad():
            macs, _ = profile(
                model,
                inputs=(dummy,),
                verbose=False,
            )

        flops = (
            float(macs) * 2
        )

        del dummy

        if model_was_training:
            model.train()

        cleanup_gpu()

    except ImportError:
        log(
            "thop not installed. "
            "FLOPs will be N/A."
        )
    except Exception as exc:
        log(
            f"FLOPs calculation failed: "
            f"{exc}"
        )

    return (
        total_params,
        trainable_params,
        size_mb,
        flops,
    )


def save_extended_metrics_txt(
    results,
    total_params,
    trainable_params,
    size_mb,
    flops,
    ece,
):
    if flops is None:
        flops_text = (
            "N/A (install thop)"
        )
    else:
        flops_text = (
            f"{flops:,.0f} "
            f"(~{flops / 1e9:.3f} GFLOPs)"
        )

    lines = [
        "OralVision - m4_coordatt_logreg EXTENDED METRICS SUMMARY",
        "=" * 70,
        f"Generated: "
        f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        "",
        "MACRO METRICS BY SPLIT",
        "-" * 70,
        (
            f"{'Split':10s} "
            f"{'N':>8s} "
            f"{'Accuracy':>10s} "
            f"{'Precision':>10s} "
            f"{'Recall':>10s} "
            f"{'F1':>10s}"
        ),
    ]

    for split, result in results.items():
        lines.append(
            f"{split:10s} "
            f"{result['n']:8d} "
            f"{result['acc']:10.4f} "
            f"{result['prec']:10.4f} "
            f"{result['rec']:10.4f} "
            f"{result['f1']:10.4f}"
        )

    lines += [
        "",
        "QWK / CHI-SQUARE DIAGNOSTIC",
        "-" * 70,
        (
            f"{'Split':10s} "
            f"{'QWK':>10s} "
            f"{'Chi2':>12s} "
            f"{'DoF':>6s} "
            f"{'p-value':>12s}"
        ),
    ]

    for split, result in results.items():
        lines.append(
            f"{split:10s} "
            f"{result['qwk']:10.4f} "
            f"{result['chi2']:12.2f} "
            f"{result['dof']:6d} "
            f"{result['p_value']:12.3e}"
        )

    lines += [
        "",
        "NOTE:",
        "QWK is retained for continuity with the original pipeline.",
        "The six disease classes are nominal, not ordinal, so QWK should",
        "not be used as the primary inferential metric.",
        "Chi-square p-values are an auxiliary diagnostic and should not",
        "be interpreted as a test that the model is clinically superior.",
        "",
        "MODEL COMPLEXITY",
        "-" * 70,
        f"Total parameters:     {total_params:,}",
        f"Trainable parameters: {trainable_params:,}",
        f"Model size on disk:   {size_mb:.2f} MB",
        f"FLOPs @ 224x224:      {flops_text}",
        "",
        "TEST CALIBRATION",
        "-" * 70,
        f"Expected Calibration Error (ECE): {ece:.6f}",
    ]

    with open(
        OUTPUT_DIR
        / "extended_metrics_summary.txt",
        "w",
        encoding="utf-8",
    ) as f:
        f.write(
            "\n".join(lines)
        )


def run_extended_evaluation(
    model,
    best_model_path,
):
    log("=" * 70)
    log(
        "EXTENDED EVALUATION: "
        "TRAIN / VAL / TEST"
    )
    log("=" * 70)

    split_datasets = {
        "train": OralDataset(
            TRAIN_SAMPLES,
            transform=eval_transform,
        ),
        "val": val_dataset,
        "test": test_dataset,
    }

    results = {}

    for split_name, dataset in (
        split_datasets.items()
    ):
        loader = make_eval_loader(
            dataset
        )

        start = time.time()

        metrics, labels, preds, probs = (
            evaluate_loader(
                model,
                loader,
            )
        )

        elapsed = (
            time.time() - start
        )

        acc = accuracy_score(
            labels,
            preds,
        )

        prec, rec, f1, _ = (
            precision_recall_fscore_support(
                labels,
                preds,
                average="macro",
                zero_division=0,
            )
        )

        qwk, chi2_stat, dof, p_value, cm_arr = (
            qwk_and_chi2(
                labels,
                preds,
            )
        )

        _, _, per_class_f1, _ = (
            precision_recall_fscore_support(
                labels,
                preds,
                average=None,
                labels=list(
                    range(NUM_CLASSES)
                ),
                zero_division=0,
            )
        )

        macro_auc = compute_macro_auc(
            labels,
            probs,
        )

        results[split_name] = {
            "n": len(labels),
            "acc": float(acc),
            "prec": float(prec),
            "rec": float(rec),
            "f1": float(f1),
            "qwk": float(qwk),
            "chi2": float(chi2_stat),
            "dof": int(dof),
            "p_value": float(p_value),
            "cm": cm_arr,
            "per_class_f1": per_class_f1,
            "macro_auc": macro_auc,
            "labels": labels,
            "preds": preds,
            "probs": probs,
        }

        log(
            f"[{split_name.upper()}] "
            f"N={len(labels)} | "
            f"accuracy={acc:.4f} | "
            f"macro_precision={prec:.4f} | "
            f"macro_recall={rec:.4f} | "
            f"macro_f1={f1:.4f} | "
            f"macro_auc={macro_auc:.4f} | "
            f"{elapsed:.1f}s"
        )

        report = classification_report(
            labels,
            preds,
            target_names=CLASS_NAMES,
            digits=4,
            zero_division=0,
        )

        report += (
            "\n\n"
            f"QWK: {qwk:.4f}\n"
            f"Chi-square statistic: "
            f"{chi2_stat:.4f}\n"
            f"DoF: {dof}\n"
            f"p-value: {p_value:.6e}\n"
        )

        with open(
            OUTPUT_DIR
            / f"{split_name}_classification_report.txt",
            "w",
            encoding="utf-8",
        ) as f:
            f.write(report)

        plt.figure(
            figsize=(8, 7)
        )

        sns.heatmap(
            cm_arr,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES,
        )

        plt.xlabel(
            "Predicted"
        )

        plt.ylabel(
            "True"
        )

        plt.title(
            f"Confusion Matrix - "
            f"{split_name.capitalize()} Set"
        )

        plt.tight_layout()

        plt.savefig(
            OUTPUT_DIR
            / f"{split_name}_confusion_matrix.png",
            dpi=150,
        )

        plt.close()

        plot_per_class_accuracy(
            cm_arr,
            CLASS_NAMES,
            split_name=split_name,
        )

        save_per_class_accuracy_txt(
            cm_arr,
            CLASS_NAMES,
            split_name=split_name,
        )

    # Combined per-class accuracy
    combined_lines = [
        "OralVision - Per-Class Accuracy "
        "(Train / Val / Test)",
        "=" * 70,
        (
            f"{'Class':22s} "
            f"{'Train':>10s} "
            f"{'Val':>10s} "
            f"{'Test':>10s}"
        ),
        "-" * 70,
    ]

    for i, class_name in enumerate(
        CLASS_NAMES
    ):
        values = []

        for split in [
            "train",
            "val",
            "test",
        ]:
            acc_values, _ = (
                per_class_accuracy_from_cm(
                    results[split]["cm"]
                )
            )
            values.append(
                acc_values[i]
            )

        combined_lines.append(
            f"{class_name:22s} "
            f"{values[0]:10.4f} "
            f"{values[1]:10.4f} "
            f"{values[2]:10.4f}"
        )

    with open(
        OUTPUT_DIR
        / "per_class_accuracy_all_splits.txt",
        "w",
        encoding="utf-8",
    ) as f:
        f.write(
            "\n".join(
                combined_lines
            )
        )

    # Model complexity
    total_params, trainable_params, size_mb, flops = (
        compute_model_complexity(
            model,
            best_model_path,
        )
    )

    plot_extended_metrics_bar(
        results
    )

    plot_qwk_pvalue_summary(
        results
    )

    plot_model_complexity_summary(
        total_params,
        trainable_params,
        size_mb,
        flops,
    )

    plot_per_class_f1_heatmap(
        results
    )

    test_probs = results[
        "test"
    ]["probs"]

    test_labels = results[
        "test"
    ]["labels"]

    test_preds = results[
        "test"
    ]["preds"]

    plot_confidence_histogram(
        test_probs,
        test_labels,
        test_preds,
    )

    ece = plot_calibration_curve(
        test_probs,
        test_labels,
        test_preds,
    )

    plot_macro_roc_auc_by_split(
        results
    )

    save_extended_metrics_txt(
        results,
        total_params,
        trainable_params,
        size_mb,
        flops,
        ece,
    )

    return (
        results,
        total_params,
        trainable_params,
        size_mb,
        flops,
        ece,
    )


# =============================================================================
# 18. Main training
# =============================================================================

def main():
    seed = CONFIG["seed"]

    seed_everything(seed)

    # -------------------------------------------------------------------------
    # Save config
    # -------------------------------------------------------------------------
    runtime_config = {
        **CONFIG,
        "device": str(DEVICE),
        "torch_version": torch.__version__,
        "torchvision_version": torchvision.__version__,
        "class_names": CLASS_NAMES,
        "num_classes": NUM_CLASSES,
    }

    with open(
        OUTPUT_DIR
        / "training_config.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            runtime_config,
            f,
            indent=2,
        )

    # -------------------------------------------------------------------------
    # Model
    # -------------------------------------------------------------------------
    log("=" * 70)
    log(
        "Creating EfficientNet-B0 + "
        "Coordinate Attention + Logistic Regression"
    )
    log("=" * 70)

    model = EfficientNetCoordAtt(
        num_classes=NUM_CLASSES,
        ca_reduction=CONFIG[
            "ca_reduction"
        ],
        pretrained=True,
    ).to(DEVICE)

    # Verify classifier is exactly one Linear layer
    assert isinstance(
        model.classifier,
        nn.Linear,
    ), (
        "Classifier head is not a single Linear layer."
    )

    log(
        f"Classifier: {model.classifier}"
    )

    save_architecture_summary(
        model
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"],
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=CONFIG["lr_factor"],
        patience=CONFIG["lr_patience"],
        min_lr=CONFIG["min_lr"],
    )

    scaler = GradScaler(
        "cuda",
        enabled=(
            CONFIG["use_amp"]
            and DEVICE.type == "cuda"
        ),
    )

    # -------------------------------------------------------------------------
    # Resume
    # -------------------------------------------------------------------------
    checkpoint = load_checkpoint()

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "train_macro_f1": [],
        "train_weighted_f1": [],
        "train_balanced_accuracy": [],

        "val_loss": [],
        "val_accuracy": [],
        "val_macro_f1": [],
        "val_weighted_f1": [],
        "val_balanced_accuracy": [],

        "lr": [],
        "epoch_time_sec": [],
    }

    best_val_f1 = -np.inf
    best_epoch = 0
    best_state = None
    patience_counter = 0
    start_epoch = 1
    total_training_time = 0.0

    if checkpoint is not None:
        log("=" * 70)
        log("RESUMING FROM CHECKPOINT")
        log("=" * 70)

        model.load_state_dict(
            checkpoint[
                "model_state_dict"
            ]
        )

        optimizer.load_state_dict(
            checkpoint[
                "optimizer_state_dict"
            ]
        )

        scheduler.load_state_dict(
            checkpoint[
                "scheduler_state_dict"
            ]
        )

        scaler.load_state_dict(
            checkpoint[
                "scaler_state_dict"
            ]
        )

        history = checkpoint[
            "history"
        ]

        best_val_f1 = checkpoint[
            "best_val_macro_f1"
        ]

        best_epoch = checkpoint[
            "best_epoch"
        ]

        best_state = checkpoint[
            "best_state"
        ]

        patience_counter = checkpoint[
            "patience_counter"
        ]

        total_training_time = checkpoint.get(
            "total_training_time",
            0.0,
        )

        difficulty_dict.clear()

        difficulty_dict.update(
            checkpoint.get(
                "difficulty_dict",
                {},
            )
        )

        try:
            set_rng_state(
                checkpoint[
                    "rng_state"
                ]
            )
            log(
                "RNG state restored successfully."
            )
        except Exception as exc:
            log(
                "WARNING: RNG restoration failed: "
                f"{exc}"
            )

        start_epoch = (
            int(
                checkpoint["epoch"]
            )
            + 1
        )

        log(
            f"Last completed epoch: "
            f"{checkpoint['epoch']}"
        )

        log(
            f"Best validation macro-F1: "
            f"{best_val_f1:.6f}"
        )

    else:
        log(
            "No checkpoint found. "
            "Starting fresh."
        )

    # -------------------------------------------------------------------------
    # Training
    # -------------------------------------------------------------------------
    if start_epoch <= CONFIG[
        "total_epochs"
    ]:
        for epoch in range(
            start_epoch,
            CONFIG["total_epochs"] + 1,
        ):
            epoch_start = time.time()

            stage = (
                "WARMUP"
                if epoch
                <= CONFIG["warmup_epochs"]
                else "CASCADE"
            )

            if (
                epoch
                == CONFIG["warmup_epochs"]
                + 1
            ):
                log("=" * 70)
                log(
                    "STAGE 2: "
                    "Hard-example cascade activated"
                )
                log("=" * 70)

            train_loader = (
                build_train_loader()
            )

            log(
                f"\nEpoch {epoch:03d}/"
                f"{CONFIG['total_epochs']} "
                f"[{stage}]"
            )

            epoch_start_train = time.time()

            train_metrics = run_train_epoch(
                model,
                train_loader,
                optimizer,
                scaler,
                epoch,
            )

            cleanup_gpu()

            val_metrics, _, _, _ = (
                evaluate_loader(
                    model,
                    val_loader,
                )
            )

            cleanup_gpu()

            scheduler.step(
                val_metrics["loss"]
            )

            current_lr = (
                optimizer.param_groups[0][
                    "lr"
                ]
            )

            epoch_time = (
                time.time()
                - epoch_start
            )

            total_training_time += (
                epoch_time
            )

            history[
                "train_loss"
            ].append(
                train_metrics["loss"]
            )

            history[
                "train_accuracy"
            ].append(
                train_metrics["accuracy"]
            )

            history[
                "train_macro_f1"
            ].append(
                train_metrics["macro_f1"]
            )

            history[
                "train_weighted_f1"
            ].append(
                train_metrics["weighted_f1"]
            )

            history[
                "train_balanced_accuracy"
            ].append(
                train_metrics[
                    "balanced_accuracy"
                ]
            )

            history[
                "val_loss"
            ].append(
                val_metrics["loss"]
            )

            history[
                "val_accuracy"
            ].append(
                val_metrics["accuracy"]
            )

            history[
                "val_macro_f1"
            ].append(
                val_metrics["macro_f1"]
            )

            history[
                "val_weighted_f1"
            ].append(
                val_metrics["weighted_f1"]
            )

            history[
                "val_balanced_accuracy"
            ].append(
                val_metrics[
                    "balanced_accuracy"
                ]
            )

            history[
                "lr"
            ].append(
                current_lr
            )

            history[
                "epoch_time_sec"
            ].append(
                epoch_time
            )

            log(
                f"[{stage}] "
                f"Epoch {epoch:03d} | "
                f"train_loss={train_metrics['loss']:.4f} | "
                f"train_acc={train_metrics['accuracy']:.4f} | "
                f"train_F1={train_metrics['macro_f1']:.4f} | "
                f"val_loss={val_metrics['loss']:.4f} | "
                f"val_acc={val_metrics['accuracy']:.4f} | "
                f"val_F1={val_metrics['macro_f1']:.4f} | "
                f"lr={current_lr:.2e} | "
                f"time={epoch_time:.1f}s"
            )

            # -------------------------------------------------------------
            # Best model
            # -------------------------------------------------------------
            if (
                val_metrics["macro_f1"]
                > best_val_f1
            ):
                best_val_f1 = (
                    val_metrics[
                        "macro_f1"
                    ]
                )

                best_epoch = epoch

                best_state = (
                    copy.deepcopy(
                        model.state_dict()
                    )
                )

                torch.save(
                    best_state,
                    OUTPUT_DIR
                    / "best_model.pth",
                )

                patience_counter = 0

                log(
                    f"  -> NEW BEST MODEL "
                    f"(val macro-F1="
                    f"{best_val_f1:.6f})"
                )
            else:
                patience_counter += 1

            # -------------------------------------------------------------
            # Save checkpoint EVERY epoch
            # -------------------------------------------------------------
            save_checkpoint(
                epoch,
                model,
                optimizer,
                scheduler,
                scaler,
                history,
                best_val_f1,
                best_epoch,
                best_state,
                patience_counter,
                total_training_time,
            )

            log(
                "Checkpoint saved."
            )

            # -------------------------------------------------------------
            # Early stopping
            # -------------------------------------------------------------
            if (
                patience_counter
                >= CONFIG[
                    "early_stop_patience"
                ]
            ):
                log(
                    f"Early stopping at epoch "
                    f"{epoch}: no validation "
                    f"macro-F1 improvement for "
                    f"{CONFIG['early_stop_patience']} epochs."
                )
                break

            print_gpu_memory(
                "End-of-epoch GPU memory: "
            )

    else:
        log(
            "Checkpoint indicates training "
            "is already complete."
        )

    # -------------------------------------------------------------------------
    # Safety: best_state must exist
    # -------------------------------------------------------------------------
    if best_state is None:
        best_model_path = (
            OUTPUT_DIR
            / "best_model.pth"
        )

        if best_model_path.exists():
            best_state = torch.load(
                best_model_path,
                map_location=DEVICE,
                weights_only=True,
            )
        else:
            raise RuntimeError(
                "No best model state is available."
            )

    # -------------------------------------------------------------------------
    # Save final model
    # -------------------------------------------------------------------------
    torch.save(
        model.state_dict(),
        OUTPUT_DIR
        / "final_model.pth",
    )

    # Load best model
    model.load_state_dict(
        best_state
    )

    model.eval()

    cleanup_gpu()

    # -------------------------------------------------------------------------
    # Final test evaluation
    # -------------------------------------------------------------------------
    log("=" * 70)
    log(
        "FINAL TEST EVALUATION "
        "USING BEST VALIDATION CHECKPOINT"
    )
    log("=" * 70)

    test_metrics, test_labels, test_preds, test_probs = (
        evaluate_loader(
            model,
            test_loader,
        )
    )

    log(
        f"Test Accuracy : "
        f"{test_metrics['accuracy']:.6f}"
    )

    log(
        f"Test Macro-F1 : "
        f"{test_metrics['macro_f1']:.6f}"
    )

    log(
        f"Test Weighted-F1 : "
        f"{test_metrics['weighted_f1']:.6f}"
    )

    log(
        f"Test Balanced Accuracy : "
        f"{test_metrics['balanced_accuracy']:.6f}"
    )

    report_txt = classification_report(
        test_labels,
        test_preds,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    )

    log("\n" + report_txt)

    with open(
        OUTPUT_DIR
        / "classification_report.txt",
        "w",
        encoding="utf-8",
    ) as f:
        f.write(
            report_txt
        )

    # -------------------------------------------------------------------------
    # Training curves
    # -------------------------------------------------------------------------
    epochs_ran = len(
        history["train_loss"]
    )

    x = np.arange(
        1,
        epochs_ran + 1,
    )

    plt.figure(
        figsize=(8, 5)
    )

    plt.plot(
        x,
        history["train_loss"],
        label="Train Loss",
    )

    plt.plot(
        x,
        history["val_loss"],
        label="Val Loss",
    )

    plt.axvline(
        CONFIG["warmup_epochs"],
        linestyle="--",
        label="Cascade starts",
    )

    plt.xlabel(
        "Epoch"
    )

    plt.ylabel(
        "Loss"
    )

    plt.title(
        "Training / Validation Loss"
    )

    plt.legend()

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / "loss_curve.png",
        dpi=150,
    )

    plt.close()

    plt.figure(
        figsize=(8, 5)
    )

    plt.plot(
        x,
        history["train_accuracy"],
        label="Train Accuracy",
    )

    plt.plot(
        x,
        history["val_accuracy"],
        label="Val Accuracy",
    )

    plt.plot(
        x,
        history["train_macro_f1"],
        linestyle=":",
        label="Train Macro-F1",
    )

    plt.plot(
        x,
        history["val_macro_f1"],
        linestyle=":",
        label="Val Macro-F1",
    )

    plt.axvline(
        CONFIG["warmup_epochs"],
        linestyle="--",
        label="Cascade starts",
    )

    plt.xlabel(
        "Epoch"
    )

    plt.ylabel(
        "Score"
    )

    plt.title(
        "Accuracy & Macro-F1"
    )

    plt.legend()

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / "accuracy_curve.png",
        dpi=150,
    )

    plt.close()

    plt.figure(
        figsize=(8, 5)
    )

    plt.plot(
        x,
        history["lr"],
    )

    plt.xlabel(
        "Epoch"
    )

    plt.ylabel(
        "Learning Rate"
    )

    plt.title(
        "ReduceLROnPlateau Schedule"
    )

    plt.yscale(
        "log"
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / "lr_curve.png",
        dpi=150,
    )

    plt.close()

    # -------------------------------------------------------------------------
    # Test confusion matrix
    # -------------------------------------------------------------------------
    test_cm = confusion_matrix(
        test_labels,
        test_preds,
        labels=list(
            range(NUM_CLASSES)
        ),
    )

    plt.figure(
        figsize=(8, 7)
    )

    sns.heatmap(
        test_cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
    )

    plt.xlabel(
        "Predicted"
    )

    plt.ylabel(
        "True"
    )

    plt.title(
        "Confusion Matrix - Test Set"
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / "confusion_matrix.png",
        dpi=150,
    )

    plt.close()

    # -------------------------------------------------------------------------
    # Test ROC curves
    # -------------------------------------------------------------------------
    y_true_bin = label_binarize(
        test_labels,
        classes=list(
            range(NUM_CLASSES)
        ),
    )

    plt.figure(
        figsize=(8, 7)
    )

    for i in range(
        NUM_CLASSES
    ):
        try:
            fpr, tpr, _ = (
                roc_curve(
                    y_true_bin[:, i],
                    test_probs[:, i],
                )
            )

            roc_auc = auc(
                fpr,
                tpr,
            )

            plt.plot(
                fpr,
                tpr,
                label=(
                    f"{CLASS_NAMES[i]} "
                    f"(AUC={roc_auc:.3f})"
                ),
            )

        except Exception:
            pass

    plt.plot(
        [0, 1],
        [0, 1],
        "k--",
        linewidth=1,
    )

    plt.xlabel(
        "False Positive Rate"
    )

    plt.ylabel(
        "True Positive Rate"
    )

    plt.title(
        "ROC Curves - Test Set"
    )

    plt.legend(
        loc="lower right",
        fontsize=8,
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / "roc_curves.png",
        dpi=150,
    )

    plt.close()

    # -------------------------------------------------------------------------
    # Legacy unprefixed test per-class files
    # -------------------------------------------------------------------------
    plot_per_class_accuracy(
        test_cm,
        CLASS_NAMES,
    )

    save_per_class_accuracy_txt(
        test_cm,
        CLASS_NAMES,
    )

    # -------------------------------------------------------------------------
    # Misclassified test grid
    # -------------------------------------------------------------------------
    plot_misclassified_grid(
        TEST_SAMPLES,
        test_labels,
        test_preds,
    )

    # -------------------------------------------------------------------------
    # Grad-CAM
    # -------------------------------------------------------------------------
    indices_by_class = {}

    for i, (_, label) in enumerate(
        TEST_SAMPLES
    ):
        indices_by_class.setdefault(
            label,
            [],
        ).append(i)

    log("=" * 70)
    log(
        "STAGE 8: Grad-CAM"
    )
    log("=" * 70)

    run_gradcam_analysis(
        model,
        TEST_SAMPLES,
        indices_by_class,
    )

    # -------------------------------------------------------------------------
    # Save H5 weights
    # -------------------------------------------------------------------------
    with h5py.File(
        OUTPUT_DIR
        / "model_weights.h5",
        "w",
    ) as hf:
        for name, tensor in (
            best_state.items()
        ):
            hf.create_dataset(
                name,
                data=tensor.detach()
                .cpu()
                .numpy(),
            )

    # -------------------------------------------------------------------------
    # Training artifacts
    # -------------------------------------------------------------------------
    artifacts = {
        "class_to_idx": CLASS_TO_IDX,
        "idx_to_class": IDX_TO_CLASS,
        "history": history,
        "config": CONFIG,
        "best_val_macro_f1": float(
            best_val_f1
        ),
        "best_epoch": int(
            best_epoch
        ),
        "test_metrics": test_metrics,
        "test_classification_report": report_txt,
        "test_confusion_matrix": test_cm.tolist(),
    }

    with open(
        OUTPUT_DIR
        / "training_artifacts.pkl",
        "wb",
    ) as f:
        pickle.dump(
            artifacts,
            f,
        )

    # -------------------------------------------------------------------------
    # Model complexity
    # -------------------------------------------------------------------------
    best_model_path = (
        OUTPUT_DIR
        / "best_model.pth"
    )

    total_params, trainable_params, size_mb, flops = (
        compute_model_complexity(
            model,
            best_model_path,
        )
    )

    # -------------------------------------------------------------------------
    # Extended evaluation
    # -------------------------------------------------------------------------
    (
        extended_results,
        _,
        _,
        _,
        _,
        ece,
    ) = run_extended_evaluation(
        model,
        best_model_path,
    )

    # -------------------------------------------------------------------------
    # Defense summary
    # -------------------------------------------------------------------------
    summary_lines = [
        "OralVision - Training Defense Summary",
        "=" * 70,
        (
            f"Generated: "
            f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
        ),
        "",
        "HARDWARE / RUNTIME",
        "-" * 70,
        f"Device: {DEVICE}",
        f"PyTorch: {torch.__version__}",
        f"Torchvision: {torchvision.__version__}",
    ]

    if torch.cuda.is_available():
        summary_lines.extend([
            f"GPU: {torch.cuda.get_device_name(0)}",
            (
                "GPU memory: "
                f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB"
            ),
        ])

    summary_lines.extend([
        "",
        "DATASET",
        "-" * 70,
        f"Dataset root: {DATASET_ROOT}",
        f"Train: {len(TRAIN_SAMPLES)}",
        f"Val:   {len(VAL_SAMPLES)}",
        f"Test:  {len(TEST_SAMPLES)}",
        f"Classes: {CLASS_NAMES}",
        "",
        "IMPORTANT DATA SPLIT NOTE",
        "-" * 70,
        "This script consumes the already-created CLEAN train/val/test split.",
        "It does not randomly re-split the dataset.",
        "It applies stochastic augmentation only to the training loader.",
        "Validation and test use deterministic ImageNet normalization only.",
        "",
        "ARCHITECTURE",
        "-" * 70,
        "EfficientNet-B0 + Coordinate Attention",
        "Classifier: Linear(1280 -> num_classes)",
        "Classifier has no dropout and no hidden layer.",
        "",
        "TRAINING",
        "-" * 70,
        (
            f"Warm-up epochs: "
            f"{CONFIG['warmup_epochs']}"
        ),
        (
            f"Cascade budget: "
            f"{CONFIG['cascade_epochs']}"
        ),
        (
            f"Maximum epochs: "
            f"{CONFIG['total_epochs']}"
        ),
        (
            f"Real batch size: "
            f"{CONFIG['batch_size']}"
        ),
        (
            f"Gradient accumulation: "
            f"x{CONFIG['gradient_accumulation_steps']}"
        ),
        (
            f"Effective batch size: "
            f"{CONFIG['effective_batch_size']}"
        ),
        (
            f"AMP: "
            f"{CONFIG['use_amp']}"
        ),
        (
            f"Class-weighted loss: "
            f"{CONFIG['use_class_weights']}"
        ),
        (
            f"Weighted sampler: "
            f"{CONFIG['use_weighted_sampler']}"
        ),
        "",
        "RESULTS",
        "-" * 70,
        (
            f"Best validation macro-F1: "
            f"{best_val_f1:.6f}"
        ),
        (
            f"Best epoch: "
            f"{best_epoch}"
        ),
        (
            f"Test accuracy: "
            f"{test_metrics['accuracy']:.6f}"
        ),
        (
            f"Test macro-F1: "
            f"{test_metrics['macro_f1']:.6f}"
        ),
        (
            f"Test weighted-F1: "
            f"{test_metrics['weighted_f1']:.6f}"
        ),
        (
            f"Test balanced accuracy: "
            f"{test_metrics['balanced_accuracy']:.6f}"
        ),
        "",
        "MODEL SIZE",
        "-" * 70,
        f"Total parameters: {total_params:,}",
        f"Trainable parameters: {trainable_params:,}",
        f"Best model size: {size_mb:.2f} MB",
        (
            f"FLOPs @ 224x224: "
            f"{flops:,.0f}"
            if flops is not None
            else "FLOPs @ 224x224: N/A"
        ),
        "",
        "CALIBRATION",
        "-" * 70,
        f"Test ECE: {ece:.6f}",
        "",
        "CHECKPOINT",
        "-" * 70,
        f"Checkpoint: {CHECKPOINT_PATH}",
        "Checkpoint is saved atomically after every completed epoch.",
        "RNG, optimizer, scheduler, AMP scaler and best state are stored.",
    ])

    with open(
        OUTPUT_DIR
        / "defense_summary.txt",
        "w",
        encoding="utf-8",
    ) as f:
        f.write(
            "\n".join(
                summary_lines
            )
        )

    # -------------------------------------------------------------------------
    # Requirements
    # -------------------------------------------------------------------------
    requirements = [
        "torch",
        "torchvision",
        "numpy",
        "pillow",
        "matplotlib",
        "seaborn",
        "scipy",
        "scikit-learn",
        "h5py",
        "thop",
    ]

    with open(
        OUTPUT_DIR
        / "requirements.txt",
        "w",
        encoding="utf-8",
    ) as f:
        f.write(
            "\n".join(
                requirements
            )
            + "\n"
        )

    # -------------------------------------------------------------------------
    # Final output
    # -------------------------------------------------------------------------
    log("=" * 70)
    log(
        "ORALVISION-M4 — RTX 2050 HARDENED RUN COMPLETE"
    )
    log("=" * 70)
    log(
        f"Best epoch: {best_epoch}"
    )
    log(
        f"Best validation macro-F1: "
        f"{best_val_f1:.6f}"
    )
    log(
        f"Test accuracy: "
        f"{test_metrics['accuracy']:.6f}"
    )
    log(
        f"Test macro-F1: "
        f"{test_metrics['macro_f1']:.6f}"
    )
    log(
        f"Outputs saved to:\n{OUTPUT_DIR}"
    )
    log("=" * 70)


if __name__ == "__main__":
    main()

[06:19:26] ======================================================================
[06:19:26] ORALVISION-M4 — RTX 2050 HARDENED VERSION
[06:19:26] ======================================================================
[06:19:26] PyTorch version : 2.5.1+cu121
[06:19:26] Torchvision     : 0.20.1+cu121
[06:19:26] Device          : cuda
[06:19:26] Dataset root    : C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN
[06:19:26] Output directory: C:\Users\R Y Z E N\Desktop\m4_coordatt_logreg_RTX2050_HARDENED
[06:19:26] Train directory : C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN\dataset\train
[06:19:26] Val directory   : C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN\dataset\val
[06:19:26] Test directory  : C:\Users\R Y Z E N\Desktop\oral_dataset_FINAL_CLEAN\dataset\test
[06:19:26] Real batch size : 16
[06:19:26] Accumulation    : x2
[06:19:26] Effective batch : 32
[06:19:26] AMP enabled     : True
[06:19:26] GPU             : NVIDIA GeForce RTX 2050
[06:19:26] GPU memory      :